# Fase 2: Feature Engineering & Text Mining (Pabrik Fitur)

## Tujuan
Menciptakan fitur prediktif dari data sensor (rolling stats, rate of change, FFT) dan mengekstrak informasi dari log teknisi.

## Langkah-langkah
### Pipeline A: Matematika Sensor
- 2A.1: Rolling Statistics (window 24 jam)
- 2A.2: Rate of Change (diff / pct_change)
- 2A.3: Ekstraksi Frekuensi (FFT)

### Pipeline B: Text Mining Log Teknisi
- 2B.1: Text Preprocessing
- 2B.2: Keyword Extraction & Hukum Konteks (Filter Preventive)
- 2B.3: Text Vectorization (Binerisasi: is_thermal, is_mechanical, is_electrical)

### Pipeline C: Data Fusion
- Multi-Key Join (machine_id + timestamp)
- Penanganan NaN pasca-join (fillna(0))

In [ ]:
import pandas as pd

# --- PRASYARAT: Data Ingestion & Cleaning (Mensimulasikan Hasil Fase 1) ---
df = pd.read_csv('../data/raw/sensor_readings.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])
df.set_index('timestamp', inplace=True)
df.interpolate(method='linear', inplace=True)
df.drop(columns=['humidity', 'operating_hours'], inplace=True)

# --- FASE 2A.1: Rolling Statistics (Jendela Berjalan) ---
print("Menghitung Rolling Statistics (mean, std, max) secara spesifik per mesin...")

# 1. Kolom numerik yang akan dihitung
sensor_cols = ['temperature', 'vibration', 'pressure', 'rpm', 'power_consumption', 'noise_level']

# Untuk menghindari Error Alignment akibat timestamp multi-mesin yang sama (contoh: M01 07:00 dan M02 07:00),
# kita WAJIB me-reset index menjadi angka unik sementara sebelum menggunakan rolling.
df.reset_index(inplace=True)

# 2 & 3. GroupBy machine_id dan buat kolom turunan Rolling 24 jam dengan penamaan yang informatif
mean_cols = [f"{col}_mean_24h" for col in sensor_cols]
std_cols  = [f"{col}_std_24h" for col in sensor_cols]
max_cols  = [f"{col}_max_24h" for col in sensor_cols]

# Operasi rolling berlandaskan grup machine_id
grouped_rolling = df.groupby('machine_id')[sensor_cols].rolling(window=24)

# Masukkan hasil perhitungan ke DataFrame asli, membersihkan label grup dengan reset_index(level=0)
df[mean_cols] = grouped_rolling.mean().reset_index(level=0, drop=True).sort_index()
df[std_cols]  = grouped_rolling.std().reset_index(level=0, drop=True).sort_index()
df[max_cols]  = grouped_rolling.max().reset_index(level=0, drop=True).sort_index()

# Kembalikan struktur murni dengan timestamp sebagai index
df.set_index('timestamp', inplace=True)

# 4. Cetak dan validasi hasil fitur
print("\n--- Daftar Susunan Kolom Setelah Rolling ---")
print(df.columns.tolist())

print("\n--- 30 Baris Pertama DataFrame ---")
display(df.head(30))

In [ ]:
# --- FASE 2A.2: Rate of Change (Kecepatan Perubahan) ---
print("Menghitung Rate of Change (Perubahan 1 Jam) per mesin...\n")

# 1. GroupBy machine_id
grouped_df = df.groupby('machine_id')

# 2 & 3. Menghitung selisih 1 jam sebelumnya (diff(1)) untuk sensor terpilih
df['temp_change_1h'] = grouped_df['temperature'].diff(1)
df['vib_change_1h'] = grouped_df['vibration'].diff(1)
df['pressure_change_1h'] = grouped_df['pressure'].diff(1)

# 4. Cetak 5 baris pertama untuk memastikan logika berjalan benar
print("--- Validasi Fitur Rate of Change ---")
display(df[['machine_id', 'temperature', 'temp_change_1h']].head(5))

In [ ]:
import numpy as np
from scipy.fft import fft, fftfreq
import warnings
warnings.filterwarnings('ignore') # Mengabaikan warning numpy saat array NaN di awal window

# --- FASE 2A.3: Ekstraksi Frekuensi (FFT) ---
print("Memulai ekstraksi Fast Fourier Transform (FFT) dari sensor getaran...")
print("Proses ini mungkin memakan waktu komputasi, mohon tunggu...\n")

# 1 & 2. Membuat fungsi Python pembantu untuk menerima array 1D window 24 jam
def get_peak_frequency(signal_window):
    # Jika window masih berisikan NaN (biasanya 23 baris pertama), lewati perhitungan
    if np.isnan(signal_window).any():
        return np.nan
    
    n = len(signal_window)
    # Sample spacing dianggap 1 (1 jam)
    yf = fft(signal_window)
    xf = fftfreq(n, 1)[:n//2]       # Mengambil frekuensi positif saja
    magnitudes = np.abs(yf[0:n//2]) # Mengambil magnitudo positif saja
    
    # Mengabaikan frekuensi 0 (DC component/rata-rata)
    magnitudes[0] = 0 
    
    # Mencari index dengan frekuensi dominan tertinggi (peak)
    peak_idx = np.argmax(magnitudes)
    return np.abs(xf[peak_idx])

# 3. Menerapkan groupby machine_id, lalu hitung FFT menggunakan rolling window 24
df.reset_index(inplace=True) # Reset sementara index timestamp seperti di 2A.1 agar rolling tidak bertabrakan

# 4. Mengeksekusi apply pada kolom vibration secara spesifik menggunakan fungsi buatan kita
df['vib_peak_freq_24h'] = df.groupby('machine_id')['vibration'].rolling(window=24).apply(get_peak_frequency, raw=True).reset_index(level=0, drop=True).sort_index()

df.set_index('timestamp', inplace=True) # Mengembalikan timestamp sebagai index

# 5. Mencetak hasil (dropna digunakan untuk membuang 23 jam pertama yang nilainya kosong karena proses windowing)
print("--- Validasi Fitur Peak Frequency (FFT) ---")
display(df[['machine_id', 'vibration', 'vib_peak_freq_24h']].dropna().head(5))

print("\n✅ Ekstraksi Matemaika Sensor (Pipeline A) Selesai!")

In [ ]:
import numpy as np

# --- PIPELINE B: TEXT MINING LOG TEKNISI ---
print("Memulai ekstraksi informasi dari catatan teks teknisi (NLP)...\n")

# 2. Memuat data log pemeliharaan
df_logs = pd.read_csv('../data/raw/maintenance_logs.csv')

print("Isi awal maintenance logs:")
display(df_logs.head(3))

# 3. Langkah 2B.1: Text Preprocessing (Ubah ke lowercase)
df_logs['technician_notes'] = df_logs['technician_notes'].str.lower()

# 4. Langkah 2B.2 & 2B.3: Keyword Extraction & Binerisasi beserta Hukum Konteks

# Syarat Hukum Konteks: Tidak boleh dari maintenance_type 'Preventive'
is_not_preventive = df_logs['maintenance_type'] != 'Preventive'

# Aturan Ekstraksi Kategori Termal
thermal_keywords = 'panas|overheat|terbakar|pendingin'
df_logs['is_thermal_issue'] = np.where(
    df_logs['technician_notes'].str.contains(thermal_keywords, case=False, na=False) & is_not_preventive, 
    1, 0
)

# Aturan Ekstraksi Kategori Mekanikal
mechanical_keywords = 'belt|bearing|bocor|aus|patah|gesek'
df_logs['is_mechanical_issue'] = np.where(
    df_logs['technician_notes'].str.contains(mechanical_keywords, case=False, na=False) & is_not_preventive, 
    1, 0
)

# Aturan Ekstraksi Kategori Elektrikal
electrical_keywords = 'sensor|short|motor|berhenti'
df_logs['is_electrical_issue'] = np.where(
    df_logs['technician_notes'].str.contains(electrical_keywords, case=False, na=False) & is_not_preventive, 
    1, 0
)

# Mengubah kolom 'date' menjadi tipe datetime dan menyimpannya sebagai kolom 'timestamp'
# agar formatnya sinkron dengan dataset sensor saat di-merge nanti.
df_logs['timestamp'] = pd.to_datetime(df_logs['date'])

# 5. Mencetak 10 baris pertama untuk divalidasi pembaca
print("\n--- Validasi Fitur Text Mining Pipeline B ---")
display(df_logs[['machine_id', 'timestamp', 'maintenance_type', 'technician_notes', 'is_thermal_issue', 'is_mechanical_issue', 'is_electrical_issue']].head(10))

print("\n✅ Ekstraksi NLP (Pipeline B) Selesai!")

In [ ]:
import re
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# 1. Inisialisasi library dan Stemmer (Asumsi pengguna sudah 'pip install Sastrawi')
print("Mempersiapkan mesin NLP bahasa Indonesia (Sastrawi)...")
factory = StemmerFactory()
stemmer = factory.create_stemmer()

# 2. Membuat stopwords manual khusus konteks teks pemeliharaan pabrik
stopwords = ["di", "ke", "dari", "yang", "karena", "untuk", "pada", "dan", "dengan", "ini", "itu"]

# 3 & 4. Membuat fungsi preprocessing formal (Pipeline 4 Lapis)
def preprocess_nlp(text):
    if not isinstance(text, str):
        return ""
    
    # Lapis 1: Case Folding (Antisipasi bila belum dilakukan)
    text = text.lower()
    
    # Lapis 2: Punctuation Removal (Buang semua tanda baca koma, titik, seru, dll)
    text = re.sub(r'[^\w\s]', '', text)
    
    # Lapis 3: Stopword Removal (Filter kata sambung)
    words = text.split()
    words = [word for word in words if word not in stopwords]
    text = " ".join(words)
    
    # Lapis 4: Stemming (Memotong imbuhan 'meng-', '-kan' untuk kembali ke akar)
    text = stemmer.stem(text)
    
    return text

# 5. Mengaplikasikan NLP ke data master log kita
print("Memulai ekstraksi tata bahasa. Proses ini mungkin memakan waktu beberapa detik...\n")
df_logs['clean_notes'] = df_logs['technician_notes'].apply(preprocess_nlp)

# 6. Mencetak dan memvalidasi log asli vs log hasil stemming yang sudah bersih
print("--- Validasi Hasil NLP Pipeline Formal ---")
display(df_logs[['technician_notes', 'clean_notes']].head(5))

In [ ]:
import numpy as np

print("Memulai Keyword Extraction dan Binerisasi dari teks yang sudah distemming...\n")

# 1. Definisi Hukum Konteks (Mengabaikan Preventive Maintenance)
is_not_preventive = df_logs['maintenance_type'] != 'Preventive'

# 2. Binerisasi untuk Isu Termal (Is Thermal Issue)
# Kata kunci disesuaikan dengan akar kata (stemming) Sastrawi: 
# 'panas' -> 'panas', 'overheat' -> 'overheat', 'terbakar' -> 'bakar', 'pendingin' -> 'dingin'
thermal_regex = 'panas|overheat|bakar|suhu|dingin'
df_logs['is_thermal_issue'] = np.where(
    df_logs['clean_notes'].str.contains(thermal_regex, case=False, na=False) & is_not_preventive, 
    1, 0
)

# 3. Binerisasi untuk Isu Mekanikal (Is Mechanical Issue)
mechanical_regex = 'belt|bearing|bocor|aus|patah|gesek'
df_logs['is_mechanical_issue'] = np.where(
    df_logs['clean_notes'].str.contains(mechanical_regex, case=False, na=False) & is_not_preventive, 
    1, 0
)

# 4. Binerisasi untuk Isu Elektrikal (Is Electrical Issue)
# 'berhenti' setelah distem akan menjadi 'henti'
electrical_regex = 'sensor|short|motor|henti'
df_logs['is_electrical_issue'] = np.where(
    df_logs['clean_notes'].str.contains(electrical_regex, case=False, na=False) & is_not_preventive, 
    1, 0
)

# 5. Mencetak hasil akhir untuk membuktikan keberhasilan binerisasi NLP
print("--- Validasi Ekstraksi Biner (Hukum Konteks Lapis AI) ---")
display(df_logs[['machine_id', 'maintenance_type', 'clean_notes', 'is_thermal_issue', 'is_mechanical_issue', 'is_electrical_issue']].head(10))

In [ ]:
# --- PIPELINE C: DATA FUSION (Penggabungan Dataset) ---
print("Memulai proses Data Fusion: Menyatukan Data Sensor & Histori Teknisi...\n")

# 1. Pastikan kolom waktu di df_logs bertipe datetime dan bernama 'timestamp' 
df_logs['timestamp'] = pd.to_datetime(df_logs['date'])

# === [PENGAMAN JUPYTER CELL RUN ULANG] ===
# Untuk mencegah terjadinya duplikasi suffix (MergeError) saat user merun ulang sel ini berkali-kali,
# pastikan kita membuang kolom hasil merge di df utama kita (bila ia mendeteksi datanya sudah ada).
cols_to_drop = ['is_thermal_issue', 'is_mechanical_issue', 'is_electrical_issue']
df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)

# Singkirkan index lama buangan jika sempat tersisa (menghindari ValueError level_0)
for cancel_col in ['level_0', 'index']:
    if cancel_col in df.columns:
        df.drop(columns=[cancel_col], inplace=True)
# =========================================

# 2. Kembalikan sementara index 'timestamp' di dataframe sensor menjadi kolom biasa 
#    agar bisa digunakan sebagai 'kunci' (key) saat proses left join.
if 'timestamp' not in df.columns:
    df.reset_index(inplace=True)

# 3. Melakukan penggabungan (Left Join)
#    Menggabungkan df (Tabel Kiri) dan df_logs (Tabel Kanan) hanya pada baris di mana
#    machine_id dan timestamp nya persis sama.
df = pd.merge(
    df, 
    df_logs[['machine_id', 'timestamp', 'is_thermal_issue', 'is_mechanical_issue', 'is_electrical_issue']], 
    on=['machine_id', 'timestamp'], 
    how='left'
)

# 4. Menerapkan hukum 'Ketiadaan Log':
#    Jika di jam/tanggal tertentu tidak ada riwayat perbaikan (NaN pasca-merge),
#    berarti dapat dipastikan secara logika tidak ada isu (0).
df[['is_thermal_issue', 'is_mechanical_issue', 'is_electrical_issue']] = df[['is_thermal_issue', 'is_mechanical_issue', 'is_electrical_issue']].fillna(0)

# 5. Kembalikan timestamp menjadi tahta aslinya (berfungsi sebagai Index Time Series)
if 'timestamp' in df.columns:
    df.set_index('timestamp', inplace=True)

# 6. Validasi hasil dan print jumlah baris untuk memastikan tidak terjadi baris meledak
print("--- Validasi Hasil Data Fusion ---")
print(f"Total Baris / Kolom Master Dataset: {df.shape}")
print("\nDistribusi Data Kolom 'is_mechanical_issue':")
display(df['is_mechanical_issue'].value_counts())

print("\n✅ Data Fusion (Pipeline C) Selesai!")

In [ ]:
# --- PENYIMPANAN HASIL FASE 2: EXPORT TO CSV ---
print("Mengekspor Data Master yang telah dilengkapi fitur engineering & NLP...")

# Menyimpan DataFrame yang menjadi kesatuan Data Fusion ke direktori 'data/processed/'
# Berfungsi sebagai jembatan yang siap dilahap oleh algoritma Anomaly Detection (Fase 3).
df.to_csv('../data/processed/sensor_features_engineered.csv')

print("✅ Selesai! Data berhasil diekspor ke '../data/processed/sensor_features_engineered.csv'")